In [33]:
import pandas as pd
import numpy as np

In [34]:
df = pd.read_csv("merged_with_opponent.csv")

In [35]:
# Sort so each team's matches are in chronological order
df = df.sort_values(['team_name', 'matchday']).reset_index(drop=True)

# For each team, grab the goals scored in that team's NEXT match
df['next_match_goals'] = df.groupby('team_name')['goals'].shift(-1)
df.head(10)

,table_position,team_name,matchday,total_shots,shots_on_target,goals,possesion,duels_won,date,opponent,opponent_goals,opponent_possession,opponent_total_shots,opponent_shots_on_target,opponent_duels_won,is_home,next_match_goals
0,2,Arsenal,1,10,4,2,0.43,0.58,2022-08-05,Crystal Palace,0.0,0.57,10.0,2.0,0.42,0,4.0
1,2,Arsenal,2,15,7,4,0.49,0.52,2022-08-13,Leicester,2.0,0.51,6.0,2.0,0.48,1,3.0
2,2,Arsenal,3,14,6,3,0.59,0.51,2022-08-20,Bournemouth,0.0,0.41,6.0,1.0,0.50,0,4.0
3,2,Arsenal,4,22,8,4,0.72,0.48,2022-08-27,Fulham,1.0,0.28,11.0,3.0,0.52,1,4.0
4,2,Arsenal,5,22,9,4,0.59,0.53,2022-08-31,Aston Villa,1.0,0.41,4.0,3.0,0.47,1,1.0
5,2,Arsenal,6,16,3,1,0.61,0.50,2022-09-04,Manchester United,3.0,0.39,10.0,6.0,0.50,0,4.0
6,2,Arsenal,7,15,7,4,0.74,0.46,2023-03-01,Everton,0.0,0.26,8.0,5.0,0.54,1,3.0
7,2,Arsenal,8,13,7,3,0.64,0.43,2022-09-18,Brentford,0.0,0.36,5.0,2.0,0.58,0,5.0
8,2,Arsenal,9,22,9,5,0.65,0.54,2022-10-01,Tottenham Hotspur,1.0,0.35,7.0,3.0,0.46,1,3.0
9,2,Arsenal,10,11,7,3,0.44,0.54,2022-10-09,Liverpool,2.0,0.56,8.0,3.0,0.46,1,1.0


In [36]:
# Shot efficiency for each individual match: goals per shot on target
df['shot_efficiency'] = df['goals'] / df['shots_on_target']

# Matches with 0 shots on target give inf/NaN -> set those to 0
df['shot_efficiency'] = df['shot_efficiency'].replace([np.inf, -np.inf], np.nan).fillna(0)

df[['team_name', 'matchday', 'goals', 'shots_on_target', 'shot_efficiency']].head(10)

,team_name,matchday,goals,shots_on_target,shot_efficiency
0,Arsenal,1,2,4,0.500000
1,Arsenal,2,4,7,0.571429
2,Arsenal,3,3,6,0.500000
3,Arsenal,4,4,8,0.500000
4,Arsenal,5,4,9,0.444444
5,Arsenal,6,1,3,0.333333
6,Arsenal,7,4,7,0.571429
7,Arsenal,8,3,7,0.428571
8,Arsenal,9,5,9,0.555556
9,Arsenal,10,3,7,0.428571


In [37]:
# --- Fix 1: fill missing opponent stats ---
# One fixture (Newcastle vs Southampton, matchday 15) is missing its score in
# the source Excel, leaving opponent_goals NaN. Any NaN here will crash some
# sklearn models downstream, so fill the opponent raw-stat columns with the
# column median (a safe, neutral value).
opp_raw_cols = [
    'opponent_goals', 'opponent_possession', 'opponent_total_shots',
    'opponent_shots_on_target', 'opponent_duels_won',
]
print('NaNs before:', df[opp_raw_cols].isna().sum().to_dict())
for col in opp_raw_cols:
    df[col] = df[col].fillna(df[col].median())
print('NaNs after: ', df[opp_raw_cols].isna().sum().to_dict())

NaNs before: {'opponent_goals': 1, 'opponent_possession': 0, 'opponent_total_shots': 0, 'opponent_shots_on_target': 0, 'opponent_duels_won': 0}
NaNs after:  {'opponent_goals': 0, 'opponent_possession': 0, 'opponent_total_shots': 0, 'opponent_shots_on_target': 0, 'opponent_duels_won': 0}


In [38]:
# --- Fix 2: opponent STRENGTH features instead of the raw opponent name ---
# The raw `opponent` string explodes into 20 one-hot columns the model can't
# learn much from. What actually matters is how strong the opponent is. Build a
# season-level strength profile per team, then attach it keyed on the opponent.
#
# strength = how that team performed across the whole season:
#   - avg goals scored (attacking strength)
#   - avg goals conceded (defensive weakness)
#   - avg shots on target faced
#   - final-ish table position (lower = stronger)
# We compute it per team from the team's own rows, then rename and merge on
# `opponent` so each row carries its opponent's strength profile.
team_strength = (
    df.groupby('team_name')
      .agg(
          str_avg_goals_scored=('goals', 'mean'),
          str_avg_shots_on_target=('shots_on_target', 'mean'),
          str_avg_possesion=('possesion', 'mean'),
          str_table_position=('table_position', 'mean'),
      )
      .reset_index()
)

# Goals conceded for a team = goals their opponents scored against them.
# In our rows, opponent_goals is exactly that, so average it per team.
conceded = (
    df.groupby('team_name')['opponent_goals'].mean()
      .rename('str_avg_goals_conceded').reset_index()
)
team_strength = team_strength.merge(conceded, on='team_name')

# Rename team_name -> opponent and prefix columns so they describe the OPPONENT
opp_strength = team_strength.rename(columns={'team_name': 'opponent'})
opp_strength.columns = ['opponent'] + [f'opp_{c}' for c in opp_strength.columns[1:]]

# Attach each row's opponent strength
df = df.merge(opp_strength, on='opponent', how='left')

print('opponent strength columns added:', [c for c in df.columns if c.startswith('opp_str')])
df[['team_name', 'opponent', 'opp_str_avg_goals_scored',
    'opp_str_avg_goals_conceded', 'opp_str_table_position']].head(10)

opponent strength columns added: ['opp_str_avg_goals_scored', 'opp_str_avg_shots_on_target', 'opp_str_avg_possesion', 'opp_str_table_position', 'opp_str_avg_goals_conceded']


,team_name,opponent,opp_str_avg_goals_scored,opp_str_avg_goals_conceded,opp_str_table_position
0,Arsenal,Crystal Palace,1.052632,1.289474,11.0
1,Arsenal,Leicester,1.342105,1.789474,18.0
2,Arsenal,Bournemouth,0.973684,1.868421,15.0
3,Arsenal,Fulham,1.447368,1.394737,10.0
4,Arsenal,Aston Villa,1.342105,1.210526,7.0
5,Arsenal,Manchester United,1.526316,1.131579,3.0
6,Arsenal,Everton,0.894737,1.500000,17.0
7,Arsenal,Brentford,1.526316,1.210526,9.0
8,Arsenal,Tottenham Hotspur,1.842105,1.657895,8.0
9,Arsenal,Liverpool,1.973684,1.236842,5.0


In [39]:
# Categorical goal class: 0-1, 2-3, 4 or more
def goal_class(g):
    if pd.isna(g):
        return np.nan
    if g <= 1:
        return '0-1'
    else:
        return '2+'

df['goal_class'] = df['next_match_goals'].apply(goal_class)
df[['team_name', 'matchday', 'next_match_goals', 'goal_class']].head(10)

,team_name,matchday,next_match_goals,goal_class
0,Arsenal,1,4.0,2+
1,Arsenal,2,3.0,2+
2,Arsenal,3,4.0,2+
3,Arsenal,4,4.0,2+
4,Arsenal,5,1.0,0-1
5,Arsenal,6,4.0,2+
6,Arsenal,7,3.0,2+
7,Arsenal,8,5.0,2+
8,Arsenal,9,3.0,2+
9,Arsenal,10,1.0,0-1


In [40]:
# Check distribution of goal class
df['goal_class'].value_counts()

goal_class
0-1    445
2+     295
Name: count, dtype: int64

In [41]:
# Get rolling averages

df['rolling_avg_goals'] = df.groupby('team_name')['goals'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['rolling_avg_efficiency'] = df.groupby('team_name')['shot_efficiency'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df[['team_name', 'matchday', 'goals', 'rolling_avg_goals', 'rolling_avg_efficiency']].head(10)

,team_name,matchday,goals,rolling_avg_goals,rolling_avg_efficiency
0,Arsenal,1,2,2.000000,0.500000
1,Arsenal,2,4,3.000000,0.535714
2,Arsenal,3,3,3.000000,0.523810
3,Arsenal,4,4,3.666667,0.523810
4,Arsenal,5,4,3.666667,0.481481
5,Arsenal,6,1,3.000000,0.425926
6,Arsenal,7,4,3.000000,0.449735
7,Arsenal,8,3,2.666667,0.444444
8,Arsenal,9,5,4.000000,0.518519
9,Arsenal,10,3,3.666667,0.470899


In [42]:
# --- Rolling form over multiple windows (last 3 and last 5 matches) ---
# These summarise a team's recent level across several stats. They are safe
# because the target is the NEXT match; everything here is known beforehand.
form_cols = ['goals', 'shots_on_target', 'total_shots', 'possesion', 'duels_won', 'shot_efficiency']

for col in form_cols:
    for window in (3, 5):
        df[f'roll{window}_{col}'] = (
            df.groupby('team_name')[col]
              .transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        )

df.filter(like='roll').head(10)

,rolling_avg_goals,rolling_avg_efficiency,roll3_goals,roll5_goals,roll3_shots_on_target,roll5_shots_on_target,roll3_total_shots,roll5_total_shots,roll3_possesion,roll5_possesion,roll3_duels_won,roll5_duels_won,roll3_shot_efficiency,roll5_shot_efficiency
0,2.000000,0.500000,2.000000,2.00,4.000000,4.000000,10.000000,10.00,0.430000,0.430000,0.580000,0.580000,0.500000,0.500000
1,3.000000,0.535714,3.000000,3.00,5.500000,5.500000,12.500000,12.50,0.460000,0.460000,0.550000,0.550000,0.535714,0.535714
2,3.000000,0.523810,3.000000,3.00,5.666667,5.666667,13.000000,13.00,0.503333,0.503333,0.536667,0.536667,0.523810,0.523810
3,3.666667,0.523810,3.666667,3.25,7.000000,6.250000,17.000000,15.25,0.600000,0.557500,0.503333,0.522500,0.523810,0.517857
4,3.666667,0.481481,3.666667,3.40,7.666667,6.800000,19.333333,16.60,0.633333,0.564000,0.506667,0.524000,0.481481,0.503175
5,3.000000,0.425926,3.000000,3.20,6.666667,6.600000,20.000000,17.80,0.640000,0.600000,0.503333,0.508000,0.425926,0.469841
6,3.000000,0.449735,3.000000,3.20,6.333333,6.600000,17.666667,17.80,0.646667,0.650000,0.496667,0.496000,0.449735,0.469841
7,2.666667,0.444444,2.666667,3.20,5.666667,6.800000,14.666667,17.60,0.663333,0.660000,0.463333,0.480000,0.444444,0.455556
8,4.000000,0.518519,4.000000,3.40,7.666667,7.000000,16.666667,17.60,0.676667,0.646000,0.476667,0.492000,0.518519,0.466667
9,3.666667,0.470899,3.666667,3.20,7.666667,6.600000,15.333333,15.40,0.576667,0.616000,0.503333,0.494000,0.470899,0.463492


In [43]:
# --- Lag features: the PREVIOUS match's values (momentum) ---
# shift(1) within each team pulls the prior match forward. The first match of
# each team has no previous game, so we backfill with the team mean to avoid NaN.
for col in ['goals', 'shots_on_target', 'shot_efficiency', 'possesion']:
    df[f'prev_{col}'] = df.groupby('team_name')[col].shift(1)
    df[f'prev_{col}'] = df[f'prev_{col}'].fillna(df.groupby('team_name')[col].transform('mean'))

df[['team_name', 'matchday', 'goals', 'prev_goals', 'prev_shots_on_target']].head(10)

,team_name,matchday,goals,prev_goals,prev_shots_on_target
0,Arsenal,1,2,2.5,5.578947
1,Arsenal,2,4,2.0,4.000000
2,Arsenal,3,3,4.0,7.000000
3,Arsenal,4,4,3.0,6.000000
4,Arsenal,5,4,4.0,8.000000
5,Arsenal,6,1,4.0,9.000000
6,Arsenal,7,4,1.0,3.000000
7,Arsenal,8,3,4.0,7.000000
8,Arsenal,9,5,3.0,7.000000
9,Arsenal,10,3,5.0,9.000000


In [44]:
# --- Season-to-date level and form trend ---
# Expanding mean = running average over all matches so far this season, a proxy
# for overall team quality. Trend = recent 3-match form minus season average,
# which is positive when a team is scoring above its own baseline (heating up).
df['season_avg_goals'] = (
    df.groupby('team_name')['goals'].transform(lambda s: s.expanding().mean())
)
df['season_avg_efficiency'] = (
    df.groupby('team_name')['shot_efficiency'].transform(lambda s: s.expanding().mean())
)

df['goals_trend'] = df['roll3_goals'] - df['season_avg_goals']
df['efficiency_trend'] = df['roll3_shot_efficiency'] - df['season_avg_efficiency']

df[['team_name', 'matchday', 'goals', 'season_avg_goals', 'goals_trend']].head(10)

,team_name,matchday,goals,season_avg_goals,goals_trend
0,Arsenal,1,2,2.000000,0.000000
1,Arsenal,2,4,3.000000,0.000000
2,Arsenal,3,3,3.000000,0.000000
3,Arsenal,4,4,3.250000,0.416667
4,Arsenal,5,4,3.400000,0.266667
5,Arsenal,6,1,3.000000,0.000000
6,Arsenal,7,4,3.142857,-0.142857
7,Arsenal,8,3,3.125000,-0.458333
8,Arsenal,9,5,3.333333,0.666667
9,Arsenal,10,3,3.300000,0.366667


In [45]:
# Quick look at the full feature set before saving
print('Total columns:', df.shape[1])
print(list(df.columns))
df.head()

Total columns: 46
['table_position', 'team_name', 'matchday', 'total_shots', 'shots_on_target', 'goals', 'possesion', 'duels_won', 'date', 'opponent', 'opponent_goals', 'opponent_possession', 'opponent_total_shots', 'opponent_shots_on_target', 'opponent_duels_won', 'is_home', 'next_match_goals', 'shot_efficiency', 'opp_str_avg_goals_scored', 'opp_str_avg_shots_on_target', 'opp_str_avg_possesion', 'opp_str_table_position', 'opp_str_avg_goals_conceded', 'goal_class', 'rolling_avg_goals', 'rolling_avg_efficiency', 'roll3_goals', 'roll5_goals', 'roll3_shots_on_target', 'roll5_shots_on_target', 'roll3_total_shots', 'roll5_total_shots', 'roll3_possesion', 'roll5_possesion', 'roll3_duels_won', 'roll5_duels_won', 'roll3_shot_efficiency', 'roll5_shot_efficiency', 'prev_goals', 'prev_shots_on_target', 'prev_shot_efficiency', 'prev_possesion', 'season_avg_goals', 'season_avg_efficiency', 'goals_trend', 'efficiency_trend']


,table_position,team_name,matchday,total_shots,shots_on_target,goals,possesion,duels_won,date,opponent,...,roll3_shot_efficiency,roll5_shot_efficiency,prev_goals,prev_shots_on_target,prev_shot_efficiency,prev_possesion,season_avg_goals,season_avg_efficiency,goals_trend,efficiency_trend
0,2,Arsenal,1,10,4,2,0.43,0.58,2022-08-05,Crystal Palace,...,0.500000,0.500000,2.5,5.578947,0.411351,0.585,2.00,0.500000,0.000000,0.000000
1,2,Arsenal,2,15,7,4,0.49,0.52,2022-08-13,Leicester,...,0.535714,0.535714,2.0,4.000000,0.500000,0.430,3.00,0.535714,0.000000,0.000000
2,2,Arsenal,3,14,6,3,0.59,0.51,2022-08-20,Bournemouth,...,0.523810,0.523810,4.0,7.000000,0.571429,0.490,3.00,0.523810,0.000000,0.000000
3,2,Arsenal,4,22,8,4,0.72,0.48,2022-08-27,Fulham,...,0.523810,0.517857,3.0,6.000000,0.500000,0.590,3.25,0.517857,0.416667,0.005952
4,2,Arsenal,5,22,9,4,0.59,0.53,2022-08-31,Aston Villa,...,0.481481,0.503175,4.0,8.000000,0.500000,0.720,3.40,0.503175,0.266667,-0.021693


In [47]:
# Drop the raw opponent NAME (replaced by opponent strength features) and the
# date string. team_name is kept for the model to one-hot encode.
df = df.drop(columns=['opponent', 'date'], errors='ignore')

# Save data
df.to_csv("cleaneddata.csv")
print('saved cleaneddata.csv', df.shape)

saved cleaneddata.csv (760, 44)
